# Lecture: REINFORCE with baseline & Actor-Critic for the InvertedPendulum environment

The **InvertedPendulum** environment from Gymnasium (MuJoCo) is a continuous-control
counterpart of CartPole: a pole is mounted on a cart and the agent must keep it
upright by applying a **continuous** horizontal force to the cart. Where CartPole
(Notebook 70) had a *discrete* action ("push left / push right"), here the action
is a real-valued torque, so the policy must output a **continuous** action.

This notebook reuses the policy-gradient idea from Notebook 70 and lifts it from a
discrete (softmax / `Categorical`) policy to a continuous (Gaussian / `Normal`)
policy. We solve the task twice, with increasing sophistication of the baseline:

1. **REINFORCE with baseline** — Monte-Carlo returns, standardised per episode.
2. **Actor-Critic (A2C)** — a *learned*, state-dependent value baseline.

## Environment Overview

- **Goal:** Keep the pole balanced upright by applying a continuous force to the cart.
- **Observation space:** 4 continuous values
  - Cart position
  - Pole angle
  - Cart velocity
  - Pole angular velocity
- **Action space:** Continuous, 1D force $\in [-3.0, 3.0]$
- **Reward:** $+1$ for every timestep the pole stays up.
- **Episode termination:**
  - The pole falls over (angle leaves the healthy range) — a **true termination**.
  - Episode length reaches 1000 steps — a **truncation** (time limit), *not* a
    termination. This distinction matters for the Actor-Critic bootstrap target.


Run the following cell only if you are working with google colab to copy the
required `.py` files into the root directory. If you are working locally just
ignore this cell!

Note: InvertedPendulum needs MuJoCo. On Colab install it with
`!pip install "gymnasium[mujoco]"`.

In [ ]:
!pip install "gymnasium[mujoco]"
!git clone https://github.com/Fjoelsak/RL.git
!cp RL/07_Policy_Search/ContinuousReinforceAgent.py ./
!cp RL/07_Policy_Search/AdvActorCriticAgent.py ./
!cp RL/07_Policy_Search/plot_utils.py ./

### Get to know the environment

### Task 1

Instantiate the environment and check the state and action space wrt type and
dimensions. What is the meaning of the shown arrays?

In [ ]:
import gymnasium as gym

env = gym.make('InvertedPendulum-v5')

# Observation space: Box of 4 continuous values (cart pos, pole angle, velocities).
print(env.observation_space)
# Action space: Box of 1 continuous value, the force applied to the cart in [-3, 3].
print(env.action_space)

### Task 2

Check the environment by sampling actions from the action space. How good is a
random agent averaged over 100 episodes? (For reference: a perfect agent scores
1000, the maximum episode length.)

In [ ]:
import numpy as np

env = gym.make('InvertedPendulum-v5')

n_eps = 100          # number of episodes to average over
reward_sum = []

for i in range(n_eps):
    obs, _ = env.reset()
    reward_per_episode = 0

    while True:
        action = env.action_space.sample()
        obs, reward, terminated, truncated, info = env.step(action)
        reward_per_episode += reward

        if terminated or truncated:
            reward_sum.append(reward_per_episode)
            break

print("Mean return of the random agent: ", np.mean(reward_sum))
env.close()

## Exercise A: Solving the problem with REINFORCE (with baseline)

As in Notebook 70, **REINFORCE is a Monte-Carlo policy-gradient method**: play a
whole episode, compute the discounted return $G_t$ from each step, and increase
the log-probability of actions in proportion to the return that followed them.

Two things change compared to the discrete CartPole agent:

- **The policy distribution.** A discrete softmax policy ($\texttt{Categorical}$)
  becomes a continuous diagonal Gaussian ($\texttt{Normal}(\mu_\theta(s), \sigma)$).
  The network outputs the mean $\mu_\theta(s)$; the spread $\sigma$ is a single
  learned, state-independent parameter. The log-probability is summed over the
  action dimensions.
- **The baseline.** We still standardise the returns per episode
  ($G_t \leftarrow (G_t - \bar G)/\sigma_G$). Subtracting the mean is a
  *variance-reducing baseline*; dividing by the std keeps the gradient scale
  constant across episodes of very different length.

The agent is implemented in `ContinuousReinforceAgent.py`.

### Task 3

Train the `ContinuousReinforceAgent` on InvertedPendulum. Inspect the config and
run the training. Because each episode yields only **one** gradient update,
REINFORCE needs a fair number of episodes to converge.

In [ ]:
# Configuration for the continuous REINFORCE (Monte Carlo Policy Gradient) agent.
config = {
    "EPISODES": 2000,        # number of training episodes (one gradient update per episode)
    "MAX_TIMESTEPS": 1000,   # cap per episode; matches InvertedPendulum's 1000-step limit
    "LEARNING_RATE": 3e-4,   # Adam step size for the policy network
    "HIDDEN_DIM": 64,        # neurons per hidden layer of the policy net
    "DISCOUNT": 0.99,        # gamma: discount factor for computing returns G_t
    "MAX_GRAD_NORM": 0.5,    # gradient clipping for stability
    "SEED": 42,              # fix all RNGs for a reproducible run (set None to disable)
    "LOGS": 'logs/reinforce_invertedpendulum/'  # stores weights + results.csv
}

In [ ]:
import gymnasium as gym
from ContinuousReinforceAgent import ContinuousReinforceAgent

env = gym.make('InvertedPendulum-v5')

reinforce_agent = ContinuousReinforceAgent(env, config)
reinforce_agent.train(env)

### Inspect the training progress

The plot shows the per-episode reward (faint) with the 100-episode moving average
on top. REINFORCE with the simple standardised-return baseline solves the task,
but you should see noticeable variance: occasional episodes collapse before the
moving average recovers — the price of a constant (state-independent) baseline.

In [ ]:
import pandas as pd
from ContinuousReinforceAgent import plot_trainingsinformation

data_REINFORCE = pd.DataFrame({
    'Rewards': reinforce_agent.reward_episodes,
    'Average score over 100 episodes': reinforce_agent.average_score_100_episodes,
})

# InvertedPendulum rewards are non-negative (+1 per step), max 1000.
plot_trainingsinformation(
    [data_REINFORCE], ['REINFORCE'], ['red'],
    columns=['Rewards', 'Average score over 100 episodes'],
    ylim=1010, ylim_low=0,
);

## Exercise B: Solving the problem with Actor-Critic (A2C)

REINFORCE's only baseline is the *constant* per-episode mean of the returns. It
reduces variance, but it cannot tell a genuinely good state from a bad one — every
step of an episode is compared against the same number.

**Actor-Critic** replaces this constant baseline with a **learned, state-dependent
value function** $V_\phi(s)$. The advantage $A_t = G_t - V_\phi(s_t)$ then measures
how much better an action was *than expected in that particular state*, which
dramatically lowers the gradient variance.

The `AdvActorCriticAgent` differs from REINFORCE in three important ways:

- **Shared Actor-Critic network** with a policy head ($\mu$, $\log\sigma$) and a
  value head $V_\phi(s)$.
- **Generalized Advantage Estimation (GAE)** instead of raw Monte-Carlo returns,
  interpolating between low-variance TD(0) and high-variance MC via `GAE_LAMBDA`.
- **Update every `N_STEPS` steps**, not once per episode. This is the key to making
  A2C work: it produces *thousands* of small updates per run. Early episodes last
  only a handful of steps, so one update per episode would give the critic far too
  little signal. Training is therefore measured in **environment steps**
  (`TOTAL_STEPS`), not episodes.

> **Termination vs. truncation.** The critic bootstraps the value target with
> $r_t + \gamma V(s_{t+1})$. This bootstrap is dropped only on a *true* termination
> (the pole fell). A time-limit truncation at 1000 steps is **not** terminal — the
> process would continue — so the agent stores `terminated` (not
> `terminated or truncated`) for the bootstrap mask.

### Task 4

Train the `AdvActorCriticAgent` on InvertedPendulum. Note the step-based config
(`TOTAL_STEPS`, `N_STEPS`) instead of the episode-based one used for REINFORCE.

In [ ]:
# Configuration for the Advantage Actor-Critic (A2C) agent.
config_a2c = {
    "TOTAL_STEPS": 200_000,  # upper bound on steps; the run auto-stops earlier (see below)
    "N_STEPS": 8,            # rollout length: one update every 8 steps
    "LEARNING_RATE": 7e-4,   # Adam step size for the shared network
    "HIDDEN_DIM": 64,        # neurons per hidden layer
    "DISCOUNT": 0.99,        # gamma: discount factor
    "GAE_LAMBDA": 0.95,      # lambda for Generalized Advantage Estimation
    "VF_COEF": 0.5,          # weight of the critic (value) loss
    "ENTROPY_COEF": 0.001,   # small entropy bonus to keep some exploration
    "STD_MIN": 0.2,          # lower bound on the policy std (slows std-collapse)
    "STD_MAX": 2.0,          # upper bound on the policy std
    "MAX_GRAD_NORM": 0.5,    # gradient clipping for stability
    "SEED": 42,              # fix all RNGs for a reproducible run (set None to disable)
    # --- collapse auto-stop ---
    "COLLAPSE_FRAC": 0.3,        # collapse = 100-ep avg drops below 30% of its best
    "COLLAPSE_MIN_SCORE": 200.0, # ...but only after the avg has reached this level
    "COLLAPSE_GRACE_STEPS": 4000,# keep training this long after detection (so it's visible)
    "LOGS": 'logs/a2c_invertedpendulum/'  # stores weights + results.csv
}

# Why a collapse auto-stop? Plain A2C has no trust region, so on long runs the
# policy eventually takes a destructive step and collapses — but *when* this
# happens is seed-dependent (early, late, or not at all within a fixed budget).
# To make the lesson reproducible we (a) fix the SEED, and (b) stop training a
# short grace period after a collapse is detected, so the collapse is always
# visible in the diagnostics plots below while the saved `best_agent` checkpoint
# still holds the good policy. Set SEED=None / COLLAPSE_FRAC=None to explore the
# raw, unguarded behaviour.

In [ ]:
import gymnasium as gym
from AdvActorCriticAgent import A2CAgent

env = gym.make('InvertedPendulum-v5')

a2c_agent = A2CAgent(env, config_a2c)
a2c_agent.train()

### Inspect the training progress

The Actor-Critic agent reaches the maximum return of 1000 and — thanks to the
learned value baseline — typically holds it more steadily than REINFORCE once it
gets there. Note the x-axis is per **episode**, but episodes early in training are
very short, so a lot of *steps* pass before the first long episodes appear.

In [ ]:
import pandas as pd
from AdvActorCriticAgent import plot_trainingsinformation

data_A2C = pd.DataFrame({
    'Rewards': a2c_agent.reward_episodes,
    'Average score over 100 episodes': a2c_agent.average_score_100_episodes,
})

plot_trainingsinformation(
    [data_A2C], ['A2C'], ['blue'],
    columns=['Rewards', 'Average score over 100 episodes'],
    ylim=1010, ylim_low=0,
);

### What to watch during training (and why it matters for PPO)

The reward curve tells you *whether* the agent is learning, but not *why* it
might suddenly break. Policy-gradient training has a few internal quantities
that are far more informative early-warning signals. The agent records them per
update in `get_diagnostics()`:

- **Policy entropy / std** — the amount of exploration. A *healthy* run lets
  these shrink slowly as the policy gets confident. If the std collapses toward
  its floor, the Gaussian becomes a spike: the log-probabilities blow up, the
  actor gradient explodes, and the policy is driven into a degenerate
  deterministic mode it cannot recover from. Watch for entropy/std falling off
  a cliff.
- **Approximate KL divergence** — how far the policy moved in a single update.
  Small, steady values mean stable improvement; a sudden spike means the update
  jumped too far and likely destroyed the policy.
- **Actor / value loss magnitude** — a jump by *orders of magnitude* (not a
  smooth change) is the collapse itself.

This is exactly the lesson that carries over to **PPO** (notebook 72). PPO's
whole purpose is to *bound* the policy update: it clips the importance ratio so
the approximate KL stays small, which is precisely the failure mode you can see
here. In practice, when training PPO you monitor the **entropy** and the
**approximate KL** every iteration — if the KL runs away or the entropy collapses,
the run is going bad regardless of what the reward is doing at that moment.

#### Reading this particular run (seed 42)

With the fixed seed the run shows a very instructive sequence — watch the plots
together rather than in isolation:

1. **Reward flat, entropy falling, KL small.** The reward barely moves for a long
   time, but the agent is *not* idle: the critic is learning a useful value
   function and the actor is committing to a rough direction (hence the slowly
   falling entropy). Falling entropy here is *healthy*, not the collapse.
2. **Occasional KL spikes while the reward is still flat.** Without a trust
   region, single updates jerk the policy around. As long as the KL falls back
   afterwards, these are productive probes — A2C feeling its way through
   parameter space.
3. **Entropy turns back up, and *then* the reward jumps.** This ordering is not a
   coincidence. The entropy rises again once the actor gradient eases off (the
   critic is now well-calibrated, so advantages shrink and the entropy bonus
   pulls the std back up). That regained exploration, sitting on top of an
   already-competent value function, is what lets the agent stumble onto the long
   balancing trajectories — and the now-accurate critic reinforces them
   immediately. So the breakthrough rides the *rising* entropy branch: it is not
   greedy narrowing that solves the task, but a restored exploration cushion on a
   solid base.
4. **Eventually: the collapse.** Later a single bad update (a large KL step) tips
   the policy into a degenerate mode it cannot escape — the reward crashes back to
   ~2. The training auto-stops a few thousand steps after this so you can see it.

> A caveat worth stating in class: this is the interpretation of one (seeded)
> run. The *qualitative* story is robust, but the exact timing of the
> breakthrough and the collapse varies across seeds — which is itself the point
> about plain A2C's fragility.

In [ ]:
import matplotlib.pyplot as plt

diag = a2c_agent.get_diagnostics()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(diag['Steps'], diag['Entropy'], color='green')
axes[0].set_title('Policy entropy'); axes[0].set_xlabel('Steps'); axes[0].grid(alpha=0.3)

axes[1].plot(diag['Steps'], diag['ApproxKL'], color='purple')
axes[1].set_title('Approximate KL per update'); axes[1].set_xlabel('Steps'); axes[1].grid(alpha=0.3)

axes[2].plot(diag['Steps'], diag['Std'], color='orange', label='policy std')
axes[2].set_title('Policy std'); axes[2].set_xlabel('Steps'); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# A quick programmatic check for the warning signs discussed above.
print(f"min entropy: {diag['Entropy'].min():.3f}   "
      f"min std: {diag['Std'].min():.3f}   "
      f"max |approx KL|: {diag['ApproxKL'].abs().max():.4f}   "
      f"max |actor loss|: {diag['ActorLoss'].abs().max():.1f}")

### Compare both agents

Plotting both moving-average curves together highlights the trade-off: REINFORCE
with a constant baseline learns but stays noisier, while the Actor-Critic's learned
state-dependent baseline gives a cleaner, more stable convergence.

In [ ]:
plot_trainingsinformation(
    [data_REINFORCE, data_A2C],
    ['REINFORCE', 'A2C'],
    ['red', 'blue'],
    columns=['Average score over 100 episodes'],
    ylim=1010, ylim_low=0,
);

### Test a trained agent

Evaluate one of the trained policies. Both agents expose `get_action(state)`, which
returns a continuous action sampled from the (now well-trained) policy. Set
`render_mode='human'` when creating the env to watch the pole balance live
(locally, not on Colab).

In [ ]:
import gymnasium as gym
import numpy as np

env = gym.make('InvertedPendulum-v5')  # add render_mode='human' to watch locally

agent = a2c_agent  # or: reinforce_agent

for ep in range(10):
    state, _ = env.reset()
    total_reward = 0
    done = False
    while not done:
        action = agent.get_action(state)
        state, reward, terminated, truncated, _ = env.step(action)
        total_reward += reward
        done = terminated or truncated
    print(f"Episode {ep}: reward {total_reward:.0f}")

env.close()

### Note: Use the following code if you are using google colab

Colab has no display for `render_mode='human'`, so instead we render the episode to
RGB frames and save them as an MP4 video that can be played inline. The video uses
the trained agent selected above.

In [ ]:
import gymnasium as gym
import imageio

env = gym.make("InvertedPendulum-v5", render_mode="rgb_array")
cur_state, _ = env.reset()

frames = []
done = False

while not done:
    frames.append(env.render())
    action = agent.get_action(cur_state)
    cur_state, reward, terminated, truncated, _ = env.step(action)
    done = terminated or truncated
    if done:
        frames.append(env.render())

env.close()

video_path = "./invertedpendulum_a2c.mp4"
imageio.mimsave(video_path, frames, fps=30)

In [ ]:
# this is for displaying the video after saving
from IPython.display import HTML
from base64 import b64encode

mp4 = open(video_path, 'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

HTML(f"""
<video width=400 controls>
    <source src="{data_url}" type="video/mp4">
</video>
""")